# Submit the Reference H2O Scoring Pipeline

Generate the native-binary batch scorer, environment, component, and CMK-compatible pipeline; then bind the reference model and golden input and inspect scored and monitoring outputs.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/05_build_and_schedule_scoring_pipeline.ipynb`.

In [ ]:
from pathlib import Path
import ast
import os
import textwrap
import yaml

from azure.ai.ml import MLClient, load_job
from azure.ai.ml.entities import ManagedIdentityConfiguration
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
bundle_value = Path(os.environ["H2O_BUNDLE_DIR"])
BUNDLE_DIR = bundle_value if bundle_value.is_absolute() else WORKSHOP_ROOT / bundle_value
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
COMPUTE_IDENTITY_CLIENT_ID = os.environ["AZUREML_COMPUTE_IDENTITY_CLIENT_ID"].strip()
if not COMPUTE_IDENTITY_CLIENT_ID:
    raise ValueError("AZUREML_COMPUTE_IDENTITY_CLIENT_ID must identify the compute cluster UMI")
OUTPUT_DATASTORE = os.getenv("AZUREML_OUTPUT_DATASTORE", "workspaceblobstore")
RUN = os.getenv("RUN_H2O_SCORING_PIPELINE", "false").lower() in {"1", "true", "yes"}

manifest = __import__("json").loads(
    (BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8")
)
generated_dir = WORKSHOP_ROOT / "outputs/generated/h2o_reference/batch"
generated_code_dir = generated_dir / "code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
score_path = generated_code_dir / "score.py"
conda_path = generated_dir / "conda.yaml"
component_path = generated_dir / "h2o-score.yaml"
pipeline_path = generated_dir / "pipeline.yaml"

score_source = r'''
import argparse
import hashlib
import json
from pathlib import Path

import h2o
import pandas as pd


def find_one(root, name):
    matches = list(Path(root).rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {name}, found {len(matches)}")
    return matches[0]


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


parser = argparse.ArgumentParser(description="Score the reference native H2O binary")
parser.add_argument("--model-dir", required=True)
parser.add_argument("--input-data", required=True)
parser.add_argument("--scored-output", required=True)
parser.add_argument("--monitoring-output", required=True)
parser.add_argument("--correlation-id", required=True)
parser.add_argument("--id-column", default="__generated__")
parser.add_argument("--fail-on-rejects", default="false")
parser.add_argument("--h2o-nthreads", type=int, default=3)
parser.add_argument("--h2o-max-mem-size", default="6G")
args = parser.parse_args()
manifest_path = find_one(args.model_dir, "model_manifest.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("model_format") != "h2o_binary":
    raise RuntimeError("Reference asset must be a native H2O binary")
if h2o.__version__ != manifest["h2o_version"]:
    raise RuntimeError(f"Expected h2o=={manifest['h2o_version']}, found {h2o.__version__}")
model_path = manifest_path.parent / manifest["model_file"]
if sha256(model_path) != manifest["files"][model_path.name]:
    raise RuntimeError("Model checksum does not match the manifest")
input_path = Path(args.input_data)
if input_path.is_dir():
    matches = list(input_path.rglob("*.csv"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one input CSV, found {len(matches)}")
    input_path = matches[0]
frame = pd.read_csv(input_path)
features = manifest["features"]
if list(frame.columns) != features:
    raise ValueError(f"Expected columns in this order: {features}")
h2o.no_progress()
h2o.init(
    ip="127.0.0.1", port=54321, start_h2o=True,
    nthreads=args.h2o_nthreads, max_mem_size=args.h2o_max_mem_size,
    strict_version_check=True, bind_to_localhost=True,
    verbose=False, telemetry=False,
)
try:
    model = h2o.load_model(str(model_path))
    h2o_frame = h2o.H2OFrame(frame)
    for column in manifest.get("categorical_features", []):
        h2o_frame[column] = h2o_frame[column].asfactor()
    prediction_frame = model.predict(h2o_frame)
    predictions = prediction_frame.as_data_frame()["predict"]
finally:
    if h2o.connection() is not None:
        h2o.cluster().shutdown(prompt=False)
scored = Path(args.scored_output)
monitoring = Path(args.monitoring_output)
scored.mkdir(parents=True, exist_ok=True)
monitoring.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"prediction": predictions}).to_csv(scored / "predictions.csv", index=False)
monitoring_frame = frame.copy()
monitoring_frame["prediction"] = predictions
monitoring_frame.to_csv(monitoring / "monitoring_data.csv", index=False)
summary = {
    "model_name": manifest["model_name"],
    "model_version": manifest["model_version"],
    "model_format": manifest["model_format"],
    "h2o_version": manifest["h2o_version"],
    "rows": len(frame),
    "correlation_id": args.correlation_id,
}
(monitoring / "summary.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(summary, indent=2))
'''
score_source = textwrap.dedent(score_source).lstrip()
ast.parse(score_source, filename=str(score_path))
score_path.write_text(score_source, encoding="utf-8")

conda_source = textwrap.dedent(f"""
name: h2o-reference-batch
channels:
  - conda-forge
dependencies:
  - python=3.12
  - openjdk=17
  - pip
  - pip:
      - h2o=={manifest['h2o_version']}
      - pandas==2.2.3
      - mlflow==2.22.1
      - azureml-mlflow==1.60.0.post1
""").lstrip()
yaml.safe_load(conda_source)
conda_path.write_text(conda_source, encoding="utf-8")

component_source = f'''$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: h2o_reference_score
display_name: Score Reference H2O Binary
type: command
inputs:
  model_dir: {{type: custom_model}}
  input_data: {{type: uri_file}}
  correlation_id: {{type: string}}
  id_column: {{type: string, default: __generated__}}
  fail_on_rejects: {{type: boolean, default: false}}
  h2o_nthreads: {{type: integer, default: 3}}
  h2o_max_mem_size: {{type: string, default: 6G}}
outputs:
  scored_output: {{type: uri_folder}}
  monitoring_output: {{type: uri_folder}}
code: ./code
environment:
  image: mcr.microsoft.com/azureml/openmpi5.0-ubuntu24.04:latest
  conda_file: ./conda.yaml
command: >-
  python score.py --model-dir ${{{{inputs.model_dir}}}}
  --input-data ${{{{inputs.input_data}}}} --scored-output ${{{{outputs.scored_output}}}}
  --monitoring-output ${{{{outputs.monitoring_output}}}}
  --correlation-id '${{{{inputs.correlation_id}}}}' --id-column '${{{{inputs.id_column}}}}'
  --fail-on-rejects ${{{{inputs.fail_on_rejects}}}}
  --h2o-nthreads ${{{{inputs.h2o_nthreads}}}}
  --h2o-max-mem-size '${{{{inputs.h2o_max_mem_size}}}}'
'''
pipeline_source = f'''$schema: https://azuremlschemas.azureedge.net/latest/pipelineJob.schema.json
type: pipeline
experiment_name: {os.environ['H2O_EXPERIMENT_NAME']}
description: Notebook-generated reference H2O scoring pipeline.
inputs:
  model_dir: {{type: custom_model, path: 'azureml:{MODEL_NAME}@latest'}}
  input_data: {{type: uri_file, path: '{(BUNDLE_DIR / 'golden_input.csv').as_posix()}'}}
  correlation_id: reference-workshop-run
  id_column: __generated__
  fail_on_rejects: false
  h2o_nthreads: 3
  h2o_max_mem_size: 6G
outputs:
  scored_output: {{mode: rw_mount}}
  monitoring_output: {{mode: rw_mount}}
settings:
  default_compute: azureml:{COMPUTE_NAME}
  default_datastore: azureml:{OUTPUT_DATASTORE}
jobs:
  score:
    type: command
    component: ./h2o-score.yaml
    inputs:
      model_dir: ${{{{parent.inputs.model_dir}}}}
      input_data: ${{{{parent.inputs.input_data}}}}
      correlation_id: ${{{{parent.inputs.correlation_id}}}}
      id_column: ${{{{parent.inputs.id_column}}}}
      fail_on_rejects: ${{{{parent.inputs.fail_on_rejects}}}}
      h2o_nthreads: ${{{{parent.inputs.h2o_nthreads}}}}
      h2o_max_mem_size: ${{{{parent.inputs.h2o_max_mem_size}}}}
    outputs:
      scored_output: ${{{{parent.outputs.scored_output}}}}
      monitoring_output: ${{{{parent.outputs.monitoring_output}}}}
'''
component_source = textwrap.dedent(component_source).lstrip()
pipeline_source = textwrap.dedent(pipeline_source).lstrip()
yaml.safe_load(component_source)
yaml.safe_load(pipeline_source)
component_path.write_text(component_source, encoding="utf-8")
pipeline_path.write_text(pipeline_source, encoding="utf-8")
print(f"Generated reference scorer: {score_path}")
print(f"Generated reference environment: {conda_path}")
print(f"Generated reference pipeline: {pipeline_path}")

job = load_job(
    pipeline_path,
    params_override=[
        {"inputs.model_dir.path": f"azureml:{MODEL_NAME}@latest"},
        {"inputs.input_data.path": str(BUNDLE_DIR / "golden_input.csv")},
        {"inputs.correlation_id": "reference-workshop-run"},
        {"inputs.id_column": "__generated__"},
        {"inputs.fail_on_rejects": False},
        {"inputs.h2o_nthreads": int(os.environ["H2O_NTHREADS"])},
        {"inputs.h2o_max_mem_size": os.environ["H2O_MAX_MEM_SIZE"]},
    ],
)
job.settings.default_compute = COMPUTE_NAME
job.settings.default_datastore = OUTPUT_DATASTORE
job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
for child_job in job.jobs.values():
    child_job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
job.display_name = "Reference H2O binary-model scoring"
job.tags = {"workshop": "azureml-h2o", "model": f"{MODEL_NAME}@latest"}

assert job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
assert all(
    child_job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
    for child_job in job.jobs.values()
)
job._validate(raise_error=True)
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")

if RUN:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Pipeline ended with status {final_job.status}")
    print({name: output.path for name, output in final_job.outputs.items()})
else:
    print(f"Prepared scoring pipeline for {MODEL_NAME}@latest on {COMPUTE_NAME}")
    print("Submission disabled. Set RUN_H2O_SCORING_PIPELINE=true in workshop/.env.")

## Expected Result

The CMK-compatible command pipeline completes on the configured cluster and publishes separate scored and monitoring output URIs.

Next: `../04_h2o_customer/01_package_and_validate_model.ipynb`.